In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/global_ecommerce_sales.csv")

df.head()
df.shape
df.info()

df.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          2000 non-null   str    
 1   Order_Date        2000 non-null   str    
 2   Customer_Name     2000 non-null   str    
 3   Customer_Segment  2000 non-null   str    
 4   Country           2000 non-null   str    
 5   Region            2000 non-null   str    
 6   Product_Category  2000 non-null   str    
 7   Product_Name      2000 non-null   str    
 8   Quantity          2000 non-null   int64  
 9   Unit_Price        2000 non-null   float64
 10  Discount_Percent  2000 non-null   int64  
 11  Total_Sales       2000 non-null   float64
 12  Shipping_Cost     2000 non-null   float64
 13  Profit            2000 non-null   float64
 14  Payment_Method    2000 non-null   str    
dtypes: float64(4), int64(2), str(9)
memory usage: 234.5 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,2000,2000,ORD-11121,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,2000,907,2023-12-11,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Name,2000,1534,Chen Patel,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Segment,2000,3,Consumer,1006,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,2000,20,Mexico,203,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,2000,5,North America,578,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Category,2000,4,Technology,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Name,2000,40,Wireless Ergonomic Mouse,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,2000.0,NaN,NaN,NaN,3.5575,2.704506,1.0,2.0,3.0,5.0,15.0
Unit_Price,2000.0,NaN,NaN,NaN,73.32949,79.311014,3.03,18.7575,46.42,90.25,472.56


In [3]:
# Missing values
df.isna().sum().sort_values(ascending=False)

df.duplicated().sum()

df["Order_Date"].nunique(), len(df)

(907, 2000)

In [4]:
categorical_columns = [
    "Customer_Segment",
    "Country",
    "Region",
    "Product_Category",
    "Payment_Method",
]

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


Customer_Segment
Customer_Segment
Consumer       1006
Corporate       623
Home Office     371
Name: count, dtype: int64

Country
Country
Mexico            203
United States     196
Canada            179
Japan             124
China             109
Australia         107
United Kingdom    106
Italy             105
France            105
Germany            95
Spain              92
India              92
South Korea        88
Argentina          66
Brazil             65
South Africa       62
Colombia           60
Saudi Arabia       53
UAE                47
Nigeria            46
Name: count, dtype: int64

Region
Region
North America           578
Asia Pacific            520
Europe                  503
Middle East & Africa    208
South America           191
Name: count, dtype: int64

Product_Category
Product_Category
Technology                567
Office Supplies           513
Furniture                 507
Clothing & Accessories    413
Name: count, dtype: int64

Payment_Method
Payment_Method
Cre

In [5]:
numeric_cols = [
    "Quantity",
    "Unit_Price",
    "Discount_Percent",
    "Total_Sales",
    "Shipping_Cost",
    "Profit"
]

for col in numeric_cols:
    print(f"\n{col}")
    print("Min   :", df[col].min())
    print("Max   :", df[col].max())
    print("Mean  :", df[col].mean())
    print("Median:", df[col].median())


Quantity
Min   : 1
Max   : 15
Mean  : 3.5575
Median: 3.0

Unit_Price
Min   : 3.03
Max   : 472.56
Mean  : 73.32948999999999
Median: 46.42

Discount_Percent
Min   : 0
Max   : 30
Mean  : 8.5725
Median: 10.0

Total_Sales
Min   : 2.42
Max   : 3813.98
Mean  : 242.27966999999998
Median: 113.125

Shipping_Cost
Min   : 5.52
Max   : 40.44
Mean  : 12.902119999999998
Median: 11.844999999999999

Profit
Min   : -11.28
Max   : 1373.63
Mean  : 79.43616
Median: 32.83


In [6]:
df["Expected_Sales"] = (
    df["Quantity"]
    * df["Unit_Price"]
    * (1 - df["Discount_Percent"] / 100)
).round(2)

df[[
    "Quantity",
    "Unit_Price",
    "Discount_Percent",
    "Total_Sales",
    "Expected_Sales"
]].head(10)

,Quantity,Unit_Price,Discount_Percent,Total_Sales,Expected_Sales
0,3,99.43,0,298.29,298.29
1,4,97.93,20,313.38,313.38
2,2,10.74,0,21.48,21.48
3,2,61.86,15,105.16,105.16
4,4,330.67,20,1058.14,1058.14
5,1,61.93,10,55.74,55.74
6,2,15.03,5,28.56,28.56
7,2,33.91,10,61.04,61.04
8,2,334.11,5,634.81,634.81
9,1,58.77,5,55.83,55.83


In [7]:
df["Sales_Difference"] = (
    df["Total_Sales"] - df["Expected_Sales"]
).round(2)

df["Sales_Difference"].value_counts().sort_index()
df.loc[
    df["Sales_Difference"] != 0, [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Percent",
        "Total_Sales",
        "Expected_Sales",
        "Sales_Difference"
    ]
]


,Order_ID,Quantity,Unit_Price,Discount_Percent,Total_Sales,Expected_Sales,Sales_Difference
24,ORD-11128,5,333.59,10,1501.15,1501.16,-0.01
72,ORD-10473,2,143.15,5,271.99,271.98,0.01
111,ORD-10124,5,11.95,10,53.77,53.78,-0.01
122,ORD-10070,5,149.77,10,673.97,673.96,0.01
227,ORD-11394,1,217.82,25,163.37,163.36,0.01
285,ORD-10674,10,14.57,5,138.41,138.42,-0.01
299,ORD-10792,5,18.79,10,84.55,84.56,-0.01
323,ORD-10961,1,61.50,15,52.27,52.28,-0.01
331,ORD-10904,1,190.35,10,171.31,171.32,-0.01
393,ORD-10917,2,12.61,25,18.91,18.92,-0.01


In [9]:

from decimal import Decimal, ROUND_HALF_UP

def calculate_expected_sales(row):
    quantity = Decimal(str(row["Quantity"]))
    unit_price = Decimal(str(row["Unit_Price"]))
    discount = Decimal(str(row["Discount_Percent"]))

    expected = quantity * unit_price * (Decimal("1") - discount / Decimal("100"))
    return expected.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

df["Expected_Sales_Decimal"] = df.apply(calculate_expected_sales, axis=1)

df["Sales_Difference_Decimal"] = df.apply(
    lambda row: Decimal(str(row["Total_Sales"])) - row["Expected_Sales_Decimal"],
    axis=1
)

sales_anomalies = df.loc[
    df["Sales_Difference_Decimal"].abs() > Decimal("0.001"),
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Percent",
        "Total_Sales",
        "Expected_Sales_Decimal",
        "Sales_Difference_Decimal"
    ]
]

display(sales_anomalies)
print("Total anomalies:", len(sales_anomalies))


,Order_ID,Quantity,Unit_Price,Discount_Percent,Total_Sales,Expected_Sales_Decimal,Sales_Difference_Decimal
19,ORD-10560,1,27.90,5,26.50,26.51,-0.01
24,ORD-11128,5,333.59,10,1501.15,1501.16,-0.01
37,ORD-10636,5,9.02,15,38.33,38.34,-0.01
52,ORD-11158,10,8.37,15,71.14,71.15,-0.01
60,ORD-10098,3,64.55,10,174.28,174.29,-0.01
...,...,...,...,...,...,...,...
1774,ORD-11046,3,9.70,5,27.64,27.65,-0.01
1917,ORD-11131,2,23.95,5,45.50,45.51,-0.01
1982,ORD-11286,2,9.45,15,16.06,16.07,-0.01
1983,ORD-11409,1,83.90,15,71.31,71.32,-0.01


Total anomalies: 69


In [10]:

# Distribusi selisih penjualan
difference_distribution = (
    df["Sales_Difference_Decimal"]
    .value_counts()
    .sort_index()
    .rename_axis("Sales_Difference")
    .reset_index(name="Transaction_Count")
)

display(difference_distribution)

# Ringkasan tingkat anomali
total_transactions = len(df)
total_anomalies = len(sales_anomalies)

print(f"Total transactions : {total_transactions}")
print(f"Total anomalies    : {total_anomalies}")
print(f"Anomaly percentage : {total_anomalies / total_transactions * 100:.2f}%")

print("\nMaximum absolute difference:")
print(df["Sales_Difference_Decimal"].abs().max())


,Sales_Difference,Transaction_Count
0,-0.01,69
1,0.00,1931


Total transactions : 2000
Total anomalies    : 69
Anomaly percentage : 3.45%

Maximum absolute difference:
0.01


In [11]:

from decimal import Decimal

def classify_sales_difference(diff):
    abs_diff = abs(diff)

    if abs_diff == Decimal("0.00"):
        return "Exact Match"
    elif abs_diff <= Decimal("0.01"):
        return "Minor Rounding Difference"
    else:
        return "Requires Investigation"

df["Sales_Validation_Status"] = (
    df["Sales_Difference_Decimal"]
    .apply(classify_sales_difference)
)

display(df["Sales_Validation_Status"].value_counts())


Sales_Validation_Status
Exact Match                  1931
Minor Rounding Difference      69
Name: count, dtype: int64

In [12]:

from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN

def calculate_sales(row, rounding_mode):
    quantity = Decimal(str(row["Quantity"]))
    price = Decimal(str(row["Unit_Price"]))
    discount = Decimal(str(row["Discount_Percent"]))

    sales = quantity * price * (
        Decimal("1") - discount / Decimal("100")
    )

    return sales.quantize(
        Decimal("0.01"),
        rounding=rounding_mode
    )

df["Sales_Half_Up"] = df.apply(
    lambda row: calculate_sales(row, ROUND_HALF_UP),
    axis=1
)

df["Sales_Half_Even"] = df.apply(
    lambda row: calculate_sales(row, ROUND_HALF_EVEN),
    axis=1
)

df["Sales_Actual_Decimal"] = df["Total_Sales"].apply(
    lambda value: Decimal(str(value))
)

comparison = pd.DataFrame({
    "Rounding_Method": ["ROUND_HALF_UP", "ROUND_HALF_EVEN"],
    "Matching_Transactions": [
        (df["Sales_Actual_Decimal"] == df["Sales_Half_Up"]).sum(),
        (df["Sales_Actual_Decimal"] == df["Sales_Half_Even"]).sum()
    ]
})

comparison["Match_Percentage"] = (
    comparison["Matching_Transactions"] / len(df) * 100
).round(2)

display(comparison)


,Rounding_Method,Matching_Transactions,Match_Percentage
0,ROUND_HALF_UP,1931,96.55
1,ROUND_HALF_EVEN,1950,97.50
